In [1]:
# Cell 0: Session Restore
# Run this cell first every session
# Loads the final modeling dataset produced by Notebook 2

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Paths
drive_root = '/content/drive/MyDrive/mets-risk-score/'
data_raw   = os.path.join(drive_root, 'data/raw/')
data_proc  = os.path.join(drive_root, 'data/processed/')
fig_dir    = os.path.join(drive_root, 'outputs/figures/')
src_dir    = os.path.join(drive_root, 'src/')

# Load modeling dataset
X = pd.read_csv(os.path.join(data_proc, 'x_modeling.csv'), low_memory=False)
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

# Load modeling dataset
X = pd.read_csv(os.path.join(data_proc, 'x_modeling.csv'), low_memory=False)
y = pd.read_csv(os.path.join(data_proc, 'y_msss.csv')).squeeze()

print('Session restored\n')
print(f'X (predictors) : {X.shape[0]:,} rows × {X.shape[1]:,} columns')
print(f'y (MSSS target): {y.shape[0]:,} rows')
print(f'y mean         : {y.mean():.3f}')
print(f'y std          : {y.std():.3f}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Session restored

X (predictors) : 1,203 rows × 435 columns
y (MSSS target): 1,203 rows
y mean         : 0.010
y std          : 1.104


In [2]:
# Cell 1: Column-level Missingness check
# Before imputation we need to understand the missingness structure:
# This audit drives every downstream decision about what goes into the MICE imputer
#  and what gets dropped before modeling.

# Missingness per column
missing_count = X.isnull().sum()
missing_pct = (missing_count / len(X) * 100).round(2)

missingness_df = pd.DataFrame({
    'column' : missing_count.index,
    'missing_n' : missing_count.values,
    'missing_pct' : missing_pct.values,
}).sort_values('missing_pct', ascending=False).reset_index(drop = True)

# Bucket columns by missingness level
complete = missingness_df[missingness_df['missing_pct'] == 0]
low = missingness_df[(missingness_df['missing_pct'] > 0) &
                     (missingness_df['missing_pct'] <= 20)]
moderate = missingness_df[(missingness_df['missing_pct'] >  20) &
                              (missingness_df['missing_pct'] <= 40)]
high = missingness_df[(missingness_df['missing_pct'] >  40) &
                              (missingness_df['missing_pct'] <  100)]
all_missing = missingness_df[missingness_df['missing_pct'] == 100]

print(f"Total columns :{X.shape[1]}")
print(f"Total Rows : {X.shape[0]:,}\n")

print(f"Complete (0% missing) :{len(complete): > 4} columns")
print(f"low (1-20% missing) :{len(low): > 4} columns")
print(f"Moderate (21-40%% missing) :{len(moderate): > 4} columns")
print(f"Hig (41-99%% missing) :{len(high): > 4} columns")
print(f"All missing :{len(all_missing): > 4} columns")

Total columns :435
Total Rows : 1,203

Complete (0% missing) :  54 columns
low (1-20% missing) : 255 columns
Moderate (21-40%% missing) :   6 columns
Hig (41-99%% missing) : 108 columns
All missing :  12 columns


In [3]:
# CELL 2: Structural Zero Fill + Feature Engineering + Missingness Drop

# Step 1: copy
X_clean = X.copy()
print(f"{X_clean.shape[0]:,} rows × {X_clean.shape[1]:,} columns")

# Step 2: Fill structural zeros for alcohol follow-ups
# Non-drinkers identified from base questions:
#   ALQ111 == 2  never had a drink in their life
#   ALQ121 == 0  never drank in past 12 months
non_drinker_mask = (X_clean['ALQ111'] == 2) | (X_clean['ALQ121'] == 0)

alcohol_followup_cols = [
    'ALQ130',  # avg drinks per day when drinking
    'ALQ142',  # days had 5+ drinks in past 12 months
    'ALQ170',  # binge drinking past 30 days
    'ALQ270',  # binge drinking frequency
    'ALQ280',  # days drunk or very high past year
    'ALQ290',  # days felt sick from drinking past year
]

# Only include columns that exist
alcohol_followup_cols = [c for c in alcohol_followup_cols
                         if c in X_clean.columns]

before_fill = X_clean[alcohol_followup_cols].isnull().sum()
X_clean.loc[non_drinker_mask, alcohol_followup_cols] = \
    X_clean.loc[non_drinker_mask, alcohol_followup_cols].fillna(0)
after_fill = X_clean[alcohol_followup_cols].isnull().sum()

print(f"\n Alcohol structural zero fill:")
print(f"   Non-drinkers identified : {non_drinker_mask.sum():,}")
for col in alcohol_followup_cols:
    filled = before_fill[col] - after_fill[col]
    print(f"   {col:<12} {before_fill[col]:>8,} "
          f"{after_fill[col]:>8,} {filled:>8,}")

# Step 3: Engineer total_MET_minutes
# Published NHANES formula for total physical activity (MET-min/week).
# Source: Global Physical Activity Questionnaire (GPAQ) methodology,
#         confirmed from NHANES PAQ documentation and published research.
#
# Formula:
#   total_MET = (days_vigorous_work   × min_vigorous_work   × 8) +
#               (days_moderate_work   × min_moderate_work   × 4) +
#               (days_walking         × min_walking         × 4) +
#               (days_vigorous_rec    × min_vigorous_rec    × 8) +
#               (days_moderate_rec    × min_moderate_rec    × 4)
#
# MET multipliers:
#   8 = vigorous activity (doubles moderate MET score)
#   4 = moderate activity
#
# Inactive people naturally get 0 - no imputation needed.
# NaN * anything = NaN, so we fill PAQ base questions with 0
# where participant indicated no activity (already confirmed complete).


# Fill PAD duration columns with 0 where base question = no activity
# Base question coding: 1 = yes, 2 = no
pad_fill_map = {
    'PAD615': ('PAQ605', 2),  # vigorous work duration - fill if no vig work
    'PAD630': ('PAQ620', 2),  # moderate work duration - fill if no mod work
    'PAD645': ('PAQ635', 2),  # walking duration       - fill if no walking
    'PAD660': ('PAQ650', 2),  # vigorous rec duration  - fill if no vig rec
    'PAD675': ('PAQ665', 2),  # moderate rec duration  - fill if no mod rec
}

for pad_col, (paq_col, no_code) in pad_fill_map.items():
    if pad_col in X_clean.columns and paq_col in X_clean.columns:
        inactive_mask = X_clean[paq_col] == no_code
        filled = X_clean.loc[inactive_mask, pad_col].isnull().sum()
        X_clean.loc[inactive_mask, pad_col] = \
            X_clean.loc[inactive_mask, pad_col].fillna(0)
        print(f"   {pad_col} ← filled {filled:,} zeros "
              f"(inactive per {paq_col})")

# Compute MET-minutes per week
# Days columns (PAQ610, PAQ625, PAQ640, PAQ655, PAQ670) * duration * MET
met_components = {
    'vigorous_work' :    ('PAQ610', 'PAD615', 8),
    'moderate_work' :    ('PAQ625', 'PAD630', 4),
    'walking'       :    ('PAQ640', 'PAD645', 4),
    'vigorous_rec'  :    ('PAQ655', 'PAD660', 8),
    'moderate_rec'  :    ('PAQ670', 'PAD675', 4),
}

X_clean['total_MET_minutes'] = 0.0

for component, (days_col, min_col, met) in met_components.items():
    if days_col in X_clean.columns and min_col in X_clean.columns:
        contribution = X_clean[days_col].fillna(0) * \
                       X_clean[min_col].fillna(0) * met
        X_clean['total_MET_minutes'] += contribution

print(f"\n   total_MET_minutes distribution:")
print(f"   Mean   : {X_clean['total_MET_minutes'].mean():.1f}")
print(f"   Median : {X_clean['total_MET_minutes'].median():.1f}")
print(f"   Min    : {X_clean['total_MET_minutes'].min():.1f}")
print(f"   Max    : {X_clean['total_MET_minutes'].max():.1f}")
print(f"   Zero   : {(X_clean['total_MET_minutes'] == 0).sum():,} "
      f"participants (completely inactive)")

# Drop individual PAQ/PAD columns now replaced by total_MET_minutes
# Keep PAD680 (sedentary minutes) — separate meaningful predictor
paq_pad_to_drop = [
    'PAQ605', 'PAQ610', 'PAQ620', 'PAQ625',
    'PAQ635', 'PAQ640', 'PAQ650', 'PAQ655',
    'PAQ665', 'PAQ670',
    'PAD615', 'PAD630', 'PAD645', 'PAD660', 'PAD675',
]
paq_pad_to_drop = [c for c in paq_pad_to_drop if c in X_clean.columns]
X_clean = X_clean.drop(columns=paq_pad_to_drop)

# Log-transform total_MET_minutes
# total_MET_minutes is right-skewed (max 125,748, 24% above WHO threshold).
# Log(1 + x) transformation compresses extreme values without dropping
# anyone. The +1 handles zero-activity participants (log(1+0) = 0).
# Standard practice in published NHANES physical activity research.

X_clean['total_MET_minutes'] = np.log1p(X_clean['total_MET_minutes'])

print(f"\n   After log(1+x) transform:")
print(f"   Mean   : {X_clean['total_MET_minutes'].mean():.3f}")
print(f"   Median : {X_clean['total_MET_minutes'].median():.3f}")
print(f"   Min    : {X_clean['total_MET_minutes'].min():.3f}")
print(f"   Max    : {X_clean['total_MET_minutes'].max():.3f}")

print(f"\n   PAQ/PAD columns dropped    : {len(paq_pad_to_drop)}")
print(f"   Replaced by                : log_total_MET_minutes + PAD680 (sedentary)")

#Step 4: Recalculate TRUE missingness
missing_count = X_clean.isnull().sum()
missing_pct   = (missing_count / len(X_clean) * 100).round(2)

missingness_df = pd.DataFrame({
    'column'      : missing_count.index,
    'missing_n'   : missing_count.values,
    'missing_pct' : missing_pct.values,
}).sort_values('missing_pct', ascending=False).reset_index(drop=True)

complete    = missingness_df[missingness_df['missing_pct'] == 0]
low         = missingness_df[(missingness_df['missing_pct'] >  0) &
                              (missingness_df['missing_pct'] <= 20)]
moderate    = missingness_df[(missingness_df['missing_pct'] >  20) &
                              (missingness_df['missing_pct'] <= 40)]
high        = missingness_df[(missingness_df['missing_pct'] >  40) &
                              (missingness_df['missing_pct'] <  100)]
all_missing = missingness_df[missingness_df['missing_pct'] == 100]

print(f"\n True Missingness After Zero-Fill & Engineering:")
print(f"    Complete  (0%)        : {len(complete):>4} columns")
print(f"    Low       (1–20%)     : {len(low):>4} columns - MICE")
print(f"    Moderate  (21–40%)    : {len(moderate):>4} columns - MICE")
print(f"    High      (41–99%)    : {len(high):>4} columns - drop")
print(f"    All missing (100%)    : {len(all_missing):>4} columns - drop")

# Step 5: Drop unusable columns
drop_log = {}

# Drop 1: 100% missing
cols_all_missing = all_missing['column'].tolist()
X_clean = X_clean.drop(columns=cols_all_missing, errors='ignore')
drop_log['100% missing'] = cols_all_missing

# Drop 2: >40% true missing
cols_high_missing = high['column'].tolist()
X_clean = X_clean.drop(columns=cols_high_missing, errors='ignore')
drop_log['>40% true missing'] = cols_high_missing

# Drop 3: Near-zero variance (>95% same value)
nzv_cols = []
for col in X_clean.select_dtypes(include=[np.number]).columns:
    top_freq = X_clean[col].value_counts(normalize=True).iloc[0]
    if top_freq >= 0.95:
        nzv_cols.append(col)
X_clean = X_clean.drop(columns=nzv_cols, errors='ignore')
drop_log['near-zero variance'] = nzv_cols

# Step 6: Drop summary
print(f"\n Drop Summary:")
total_dropped = 0
for reason, cols in drop_log.items():
    print(f"   {reason:<25} : {len(cols):>4} columns dropped")
    total_dropped += len(cols)

print(f"\n   Total columns dropped  : {total_dropped}")
print(f"   Columns remaining      : {X_clean.shape[1]}")
print(f"   Rows unchanged         : {X_clean.shape[0]:,}")

# Show high-missing columns dropped for transparency
if cols_high_missing:
    print(f"\n   Columns dropped (>40% true missing):")
    for col in cols_high_missing[:20]:  # show top 20
        pct = missingness_df[
            missingness_df['column'] == col
        ]['missing_pct'].values
        if len(pct) > 0:
            print(f"   {col:<30} {pct[0]:>9.1f}%")
    if len(cols_high_missing) > 20:
        print(f"   ... and {len(cols_high_missing)-20} more")

print(f"\n {X_clean.shape[1]} columns remaining")

1,203 rows × 435 columns

 Alcohol structural zero fill:
   Non-drinkers identified : 92
   ALQ130            241      149       92
   ALQ142            241      149       92
   ALQ170            241      149       92
   ALQ270            670      578       92
   ALQ280            670      578       92
   ALQ290            997      905       92
   PAD615 ← filled 771 zeros (inactive per PAQ605)
   PAD630 ← filled 553 zeros (inactive per PAQ620)
   PAD645 ← filled 882 zeros (inactive per PAQ635)
   PAD660 ← filled 732 zeros (inactive per PAQ650)
   PAD675 ← filled 660 zeros (inactive per PAQ665)

   total_MET_minutes distribution:
   Mean   : 6895.0
   Median : 3240.0
   Min    : 0.0
   Max    : 125748.0
   Zero   : 185 participants (completely inactive)

   After log(1+x) transform:
   Mean   : 7.000
   Median : 8.084
   Min    : 0.000
   Max    : 11.742

   PAQ/PAD columns dropped    : 15
   Replaced by                : log_total_MET_minutes + PAD680 (sedentary)

 True Missingness Aft

We initially attempted MICE via miceforest (random forest imputation),
which is theoretically more flexible for mixed-domain data like ours
(biochemistry, dietary, lifestyle, behavioral variables with non-linear
relationships). However, random forest MICE builds a separate model for
each missing-value column per dataset per iteration — on a 285-column
dataset with n=5 and 5 iterations, this means constructing and holding
thousands of LightGBM models in memory simultaneously. This consistently
exhausted available RAM in the Google Colab environment regardless of
parameter optimization (reduced n_estimators, max_depth, num_leaves,
save_models=0, and sequential single-dataset execution).

We therefore switched to sklearn's IterativeImputer with BayesianRidge
as the imputation estimator. This is the published standard for MICE
in clinical epidemiology literature, is sklearn's own default imputer,
and is widely accepted by reviewers at journals such as PLOS ONE and
BMC Public Health. Critically, our overall missingness is 6.4%
(22,226 missing values out of 343,755 total) — at this level, the
difference in imputation accuracy between random forest and BayesianRidge
is negligible for downstream modeling. Multiple simulation studies have
shown that imputation method choice has minimal impact on prediction
model performance when missingness is below 10-15% (Sterne et al., 2009;
van Buuren, 2018). The n=5 multiple imputation framework is preserved —
five independent datasets are generated using different random seeds,
allowing us to report model performance consistency across all five
imputed datasets in the methods section.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2b │ Dietary Column Preprocessing
# ───────────────────────────────────────────────────────────────────────────
# Three-step process to make dietary columns stable for imputation:
#
# Step 1: Cap DR1T columns at 99th percentile
#         A single extreme outlier (DR1TKCAL=10,477 calories) caused
#         BayesianRidge overflow. Capping removes extreme values while
#         preserving 99% of the real distribution.
#
# Step 2: Log-transform all dietary columns with log(1+x)
#         Dietary nutrient columns are heavily right-skewed (skew 1-20).
#         BayesianRidge assumes near-normal distributions — extreme skewness
#         causes numerical overflow even after capping. Log(1+x) is standard
#         practice in nutritional epidemiology for exactly this reason.
#
# Step 3: Drop columns with skew >3 after double log-transform
#         Eight specific fatty acid subcomponents remained extreme after
#         both single and double log-transform. Signal retained via
#         DR1TPFAT/DR2TPFAT (total polyunsaturated fat, still in X_clean).
# ═══════════════════════════════════════════════════════════════════════════

# ── Step 1: Cap DR1T columns at 99th percentile ─────────────────────────────
dr1t_cols = [c for c in X_clean.columns if c.startswith('DR1T')]

capped = 0
for col in dr1t_cols:
    cap_val = X_clean[col].dropna().quantile(0.99)
    n       = (X_clean[col] > cap_val).sum()
    if n > 0:
        X_clean[col] = X_clean[col].clip(upper=cap_val)
        capped += 1

print(f"✅ Step 1: Capped {capped} DR1T columns at 99th percentile")

# ── Step 2: Log-transform all dietary columns ───────────────────────────────
dietary_cols = [c for c in X_clean.columns
                if c.startswith('DR1T') or c.startswith('DR2T')]

for col in dietary_cols:
    X_clean[col] = np.log1p(X_clean[col])

print(f"✅ Step 2: Log-transformed {len(dietary_cols)} dietary columns")

# ── Step 3: Drop columns with skew >3 after transformation ──────────────────
cols_to_drop = [
    # >80% zeros after transform — near-zero variance
    'DR1TATOA', 'DR2TM221', 'DR2TALCO',
    # Remain heavily skewed (4-8) after double log — unstable for imputation
    # Signal retained via DR1TPFAT/DR2TPFAT (total polyunsaturated fat)
    'DR1TP184', 'DR1TP205',
    'DR2TP184', 'DR2TP205', 'DR2TP225', 'DR2TP226',
]
cols_to_drop = [c for c in cols_to_drop if c in X_clean.columns]
X_clean      = X_clean.drop(columns=cols_to_drop)

print(f"✅ Step 3: Dropped {len(cols_to_drop)} high-skew dietary columns")
print(f"\n   Final shape    : {X_clean.shape[0]:,} rows × {X_clean.shape[1]:,} columns")
print(f"   Missing values : {X_clean.isnull().sum().sum():,}")

# ── Verify no extreme skew remains ──────────────────────────────────────────
remaining = [(c, X_clean[c].dropna().skew())
             for c in X_clean.columns
             if c.startswith('DR') and abs(X_clean[c].dropna().skew()) > 3]

if remaining:
    print(f"\n⚠️  Still high-skew:")
    for col, skew in remaining:
        print(f"   {col:<20} skew={skew:.2f}")
else:
    print(f"\n✅ No high-skew dietary columns remaining — ready for MICE")

✅ Step 1: Capped 66 DR1T columns at 99th percentile
✅ Step 2: Log-transformed 134 dietary columns
✅ Step 3: Dropped 9 high-skew dietary columns

   Final shape    : 1,203 rows × 281 columns
   Missing values : 20,974

⚠️  Still high-skew:
   DR1LANG              skew=3.06
   DR1STY               skew=5.41
   DRQSDIET             skew=5.46
   DR1TP226             skew=3.24
   DR1_300              skew=3.63
   DRD340               skew=4.95
   DRD360               skew=5.90
   DR2TATOA             skew=3.28
   DR2_330Z             skew=3.55


In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2c FIXED │ Drop Problematic Columns
# ───────────────────────────────────────────────────────────────────────────
# Two categories of columns to drop:
#
# 1. Categorical dietary flags (3-4 unique values) — these are interview
#    condition codes, not nutritional measures. Log-transform amplified
#    their skew artificially. No meaningful signal for MSSS prediction.
#
# 2. String time columns — SLQ300/310/320/330 are clock times (HH:MM)
#    that survived session restore. Already redundant with SLD012/SLD013.
# ═══════════════════════════════════════════════════════════════════════════

# ── Drop categorical dietary flag columns ───────────────────────────────────
categorical_drop = ['DR1LANG', 'DR1STY', 'DRQSDIET',
                    'DR1_300', 'DRD340', 'DRD360']

# ── Drop string time columns if still present ───────────────────────────────
string_drop = ['SLQ300', 'SLQ310', 'SLQ320', 'SLQ330', 'BPAOARM']

all_drop = categorical_drop + string_drop
all_drop = [c for c in all_drop if c in X_clean.columns]

X_clean = X_clean.drop(columns=all_drop)

print(f"🗑️  Dropped {len(all_drop)} columns:")
for col in all_drop:
    print(f"   {col}")

# ── Final skew check — numeric columns only ─────────────────────────────────
numeric_cols = X_clean.select_dtypes(include=[np.number]).columns

high_skew = [
    (col, X_clean[col].dropna().skew())
    for col in numeric_cols
    if abs(X_clean[col].dropna().skew()) > 3
]

print(f"\n📊 Final state:")
print(f"   Shape          : {X_clean.shape[0]:,} rows × {X_clean.shape[1]:,} columns")
print(f"   Missing values : {X_clean.isnull().sum().sum():,}")
print(f"   Object columns : {X_clean.select_dtypes(include=['object']).shape[1]}")
print(f"   High-skew (>3) : {len(high_skew)} columns")

if high_skew:
    print(f"\n   Remaining high-skew columns:")
    for col, skew in sorted(high_skew,
                            key=lambda x: abs(x[1]),
                            reverse=True)[:10]:
        print(f"   {col:<20} skew={skew:.2f}")
    if len(high_skew) > 10:
        print(f"   ... and {len(high_skew)-10} more")
else:
    print(f"\n✅ No high-skew columns — ready for MICE")

🗑️  Dropped 11 columns:
   DR1LANG
   DR1STY
   DRQSDIET
   DR1_300
   DRD340
   DRD360
   SLQ300
   SLQ310
   SLQ320
   SLQ330
   BPAOARM

📊 Final state:
   Shape          : 1,203 rows × 270 columns
   Missing values : 20,610
   Object columns : 0
   High-skew (>3) : 29 columns

   Remaining high-skew columns:
   DMDBORN4             skew=33.60
   ALQ130               skew=32.20
   PAD680               skew=18.68
   LBXSCR               skew=18.10
   LBDSCRSI             skew=18.10
   LBXSCK               skew=17.76
   ALQ170               skew=16.10
   LBXNRBC              skew=13.22
   DBQ095Z              skew=9.11
   LBXSGTSI             skew=8.65
   ... and 19 more


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3c FIXED │ MICE with Post-Imputation Clipping
# ───────────────────────────────────────────────────────────────────────────
# BayesianRidge occasionally extrapolates outside the observed range
# for columns with tight distributions and high missingness correlation.
# We clip each imputed column to its observed min/max range immediately
# after imputation — preventing impossible values while preserving the
# imputed distribution shape.
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import time
import gc
import os

# ── Store observed min/max for all columns before imputation ────────────────
col_bounds = {}
for col in X_clean.columns:
    observed = X_clean[col].dropna()
    if len(observed) > 0:
        col_bounds[col] = (observed.min(), observed.max())

print(f"📋 Pre-imputation confirmation:")
print(f"   Shape          : {X_clean.shape[0]:,} rows × {X_clean.shape[1]:,} columns")
print(f"   Missing values : {X_clean.isnull().sum().sum():,}")
print(f"   Column bounds  : stored for {len(col_bounds)} columns")
print(f"\n⏳ Running MICE with post-imputation clipping...\n")

total_start   = time.time()
imputed_paths = []

for i in range(5):
    iter_start = time.time()
    print(f"   Dataset {i+1}/5 — running...", end=" ")

    # ── Run imputer ─────────────────────────────────────────────────────────
    imputer = IterativeImputer(
        estimator    = BayesianRidge(),
        max_iter     = 10,
        random_state = 42 + i,
        verbose      = 0
    )

    imputed_array = imputer.fit_transform(X_clean)
    imputed_df    = pd.DataFrame(imputed_array, columns=X_clean.columns)

    # ── Clip to observed range ───────────────────────────────────────────────
    for col, (col_min, col_max) in col_bounds.items():
        if col in imputed_df.columns:
            imputed_df[col] = imputed_df[col].clip(lower=col_min, upper=col_max)

    # ── Verify ──────────────────────────────────────────────────────────────
    missing_left = imputed_df.isnull().sum().sum()

    # ── Save ────────────────────────────────────────────────────────────────
    save_path = os.path.join(data_proc, f'x_imputed_{i+1}.csv')
    imputed_df.to_csv(save_path, index=False)
    size_mb   = os.path.getsize(save_path) / (1024 * 1024)
    imputed_paths.append(save_path)

    iter_elapsed = (time.time() - iter_start) / 60
    print(f"✅ {iter_elapsed:.1f} min | missing={missing_left} | {size_mb:.1f} MB")

    del imputer, imputed_array, imputed_df
    gc.collect()

total_elapsed = (time.time() - total_start) / 60
print(f"\n✅ All 5 datasets complete in {total_elapsed:.1f} minutes")
print(f"   Saved: x_imputed_1.csv through x_imputed_5.csv")

📋 Pre-imputation confirmation:
   Shape          : 1,203 rows × 270 columns
   Missing values : 20,610
   Column bounds  : stored for 270 columns

⏳ Running MICE with post-imputation clipping...

   Dataset 1/5 — running... ✅ 4.7 min | missing=0 | 3.9 MB
   Dataset 2/5 — running... ✅ 4.8 min | missing=0 | 3.9 MB
   Dataset 3/5 — running... ✅ 4.8 min | missing=0 | 3.9 MB
   Dataset 4/5 — running... ✅ 4.8 min | missing=0 | 3.9 MB
   Dataset 5/5 — running... ✅ 4.8 min | missing=0 | 3.9 MB

✅ All 5 datasets complete in 23.9 minutes
   Saved: x_imputed_1.csv through x_imputed_5.csv


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3d │ Imputation Quality Check
# ═══════════════════════════════════════════════════════════════════════════

print("📊 Imputation Quality Check")
print("   Observed vs Imputed distributions (Dataset 1)\n")

imputed_1 = pd.read_csv(os.path.join(data_proc, 'x_imputed_1.csv'))

check_cols = {
    'Biochemistry'  : 'LBXSATSI',
    'CBC'           : 'LBXWBCSI',
    'Dietary'       : 'DR1TKCAL',
    'Sleep'         : 'SLD012',
    'Mental Health' : 'DPQ010',
    'Alcohol'       : 'ALQ130',
    'Physical Act.' : 'total_MET_minutes',
}

print(f"   {'Domain':<15} {'Column':<22} "
      f"{'Obs Mean':>10} {'Imp Mean':>10} "
      f"{'Obs Std':>9} {'Imp Std':>9}  Match")
print('   ' + '─' * 82)

for domain, col in check_cols.items():
    if col not in X_clean.columns:
        print(f"   {domain:<15} {col:<22} ← not in X_clean")
        continue

    obs       = X_clean[col].dropna()
    imp       = imputed_1[col]
    mean_diff = abs(obs.mean() - imp.mean()) / abs(obs.mean()) * 100
    match     = "✅" if mean_diff < 5 else "⚠️  check"

    print(f"   {domain:<15} {col:<22} "
          f"{obs.mean():>10.3f} {imp.mean():>10.3f} "
          f"{obs.std():>9.3f} {imp.std():>9.3f}  "
          f"{match} ({mean_diff:.1f}% diff)")

print(f"\n📁 Verifying all 5 datasets:")
for i in range(5):
    path    = os.path.join(data_proc, f'x_imputed_{i+1}.csv')
    df      = pd.read_csv(path)
    missing = df.isnull().sum().sum()
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"   x_imputed_{i+1}.csv : "
          f"{df.shape[0]:,} rows × {df.shape[1]:,} cols | "
          f"missing={missing} | {size_mb:.1f} MB ✅")

print(f"\n✅ Quality check complete")

📊 Imputation Quality Check
   Observed vs Imputed distributions (Dataset 1)

   Domain          Column                   Obs Mean   Imp Mean   Obs Std   Imp Std  Match
   ──────────────────────────────────────────────────────────────────────────────────
   Biochemistry    LBXSATSI                   22.907     22.895    21.878    21.862  ✅ (0.1% diff)
   CBC             LBXWBCSI                    6.768      6.768     2.009     2.009  ✅ (0.0% diff)
   Dietary         DR1TKCAL                    7.594      7.466     0.525     0.856  ✅ (1.7% diff)
   Sleep           SLD012                      7.525      7.524     1.730     1.728  ✅ (0.0% diff)
   Mental Health   DPQ010                      0.378      0.374     0.725     0.715  ✅ (0.8% diff)
   Alcohol         ALQ130                      3.580      3.631    30.775    28.810  ✅ (1.4% diff)
   Physical Act.   total_MET_minutes           7.000      7.000     3.246     3.246  ✅ (0.0% diff)

📁 Verifying all 5 datasets:
   x_imputed_1.csv : 1,2